# Fine-Tuning Ultra-Rapido con Unsloth y Exportacion a GGUF para Ollama

**Nivel:** Avanzado  
**Tecnologias:** `unsloth`, OpenAI Triton, PyTorch, GGUF, Ollama  
**Modelo Base:** Google Gemma 2 2B Instruct 4-bit (`unsloth/gemma-2-2b-it-bnb-4bit`)  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/03-fast-finetuning-unsloth-gguf/03_fast_finetuning_unsloth_gguf.ipynb)

---

## 1. Fundamentos Tecnologicos: Por que Unsloth y que es GGUF?

### La Arquitectura de Unsloth
**Unsloth** es un framework open-source de optimizacion de LLMs que reescribe los kernels computacionales de PyTorch directamente en **OpenAI Triton**:
- **Kernels Manuales:** Reemplaza multiplicaciones de atencion, RoPE (Rotary Position Embeddings), Cross-Entropy y RMSNorm por versiones compiladas a bajo nivel.
- **Backpropagation Manual:** Calcula derivadas analiticas intermedias evitando almacenar mapas de activacion masivos en VRAM.
- **Rendimiento:** 2x a 5x mayor velocidad de entrenamiento y hasta un **70% de ahorro en consumo de memoria GPU**, con exactamente 0% de perdida en precision numerica.

### El Formato Binario GGUF y el Ecosistema de Serving (Ollama / vLLM)
En entornos productivos, servir un modelo cargando checkpoints de PyTorch de 16 bits en Hugging Face es costoso e ineficiente. El formato **GGUF (Georgi Gerganov Unified Format)** es el estandar universal de `llama.cpp`:
- Almacena metadatos y tensores cuantizados en un unico archivo binario compacto.
- Permite mapeo de memoria directo (*mmap*) y offloading parcial entre CPU y GPU.
- Es el formato nativo consumido por motores de inferencia de alto rendimiento como **Ollama**, **vLLM**, **LM Studio** y **LiteRT**.

### Objetivos de este Laboratorio
1. Cargar un modelo pre-cuantizado a 4 bits con `FastLanguageModel` de Unsloth.
2. Inyectar adaptadores LoRA en todos los modulos lineales (`q`, `k`, `v`, `o`, `gate`, `up`, `down`).
3. Entrenar el modelo en una tarea de **extraccion estructurada en formato JSON estricto**.
4. Evaluar la inferencia acelerada token por token.
5. Exportar el modelo resultante directamente a formato cuantizado **GGUF** (`q4_k_m`).
6. Generar el `Modelfile` para servir el modelo inmediatamente con **Ollama** (puente hacia la Sesion 4).


### Paso 1: Instalacion Especializada de Unsloth en Google Colab (GPU CUDA)

Instalamos Unsloth y las bibliotecas complementarias de entrenamiento:


Si lo haces fuera de codelab

**pip install -q --no-deps trl peft accelerate bitsandbytes datasets**

In [1]:
!pip install -q unsloth
!pip install -q "transformers==4.57.6" "huggingface_hub>=0.34,<1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

In [2]:
import trl, transformers
print(trl.__version__, transformers.__version__)
from trl import SFTTrainer

0.24.0 4.57.6


In [5]:
import json
import math
import torch
import transformers
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1554: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers: {transformers.__version__}")

if int(transformers.__version__.split(".")[0]) >= 5:
    raise SystemExit(
        "\ntransformers 5.x detectado. Rompe la carga de checkpoints -bnb-4bit\n"
        "(capas inicializadas al azar, sin error visible).\n"
        '  !pip install -q "transformers==4.57.6" "huggingface_hub>=0.34,<1.0"\n'
        "y reinicia la sesion.\n"
    )

if not torch.cuda.is_bf16_supported():
    raise SystemExit(
        "\nGPU sin bf16 (Turing o anterior, p.ej. T4). Gemma-2 tiene\n"
        "activaciones que superan el maximo de fp16 (65504) -> inf -> NaN.\n"
        "Cambia a L4/A100, o usa 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'.\n"
    )

GPU: NVIDIA L4
transformers: 4.57.6


### Paso 2: Carga Optimizada del Modelo Base con `FastLanguageModel`

Cargamos `unsloth/gemma-2-2b-it-bnb-4bit` con soporte de longitud de contexto de hasta 2048 tokens consumiendo menos de 4 GB de VRAM:


In [7]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

MAX_SEQ_LENGTH = 512
MODELO = "unsloth/gemma-2-2b-it-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

assert tokenizer.chat_template is not None, "El tokenizer no trae chat_template"

print("Modelo y tokenizador de Unsloth cargados exitosamente.")

Unsloth: If you want to finetune Gemma 2, install flash-attn to make it faster!
To install flash-attn, do the below:

pip install --no-deps --upgrade "flash-attn>=2.6.3"
==((====))==  Unsloth 2026.9.4: Fast Gemma2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Modelo y tokenizador de Unsloth cargados exitosamente.


In [8]:
# Prueba que gemma responde bien

_t = "The capital of France is Paris. It is a large city in Europe."
_e = tokenizer(_t, return_tensors="pt").to("cuda")
with torch.no_grad():
    _loss = model(**_e, labels=_e.input_ids).loss.item()
print(f"\nSalud del modelo base: loss = {_loss:.3f}  (sano: 2-4)")
if _loss > 12:
    raise SystemExit(
        f"Modelo corrupto al cargar (loss {_loss:.2f} sobre texto trivial).\n"
        "Prueba con MODELO = 'unsloth/gemma-2-2b-it' (sin -bnb-4bit),\n"
        "manteniendo load_in_4bit=True para que se cuantice en el momento.\n"
    )


Salud del modelo base: loss = 2.719  (sano: 2-4)


### Paso 3: Inyeccion de Adaptadores LoRA en Todos los Modulos Lineales

A diferencia de SFT basico que solo entrena `q_proj` y `v_proj`, Unsloth permite entrenar de forma eficiente **todos los modulos lineales** sin impacto severo en VRAM, maximizando la capacidad de adaptacion:


In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("Adaptadores LoRA inyectados con kernels optimizados de Unsloth.")

Unsloth 2026.9.4 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Adaptadores LoRA inyectados con kernels optimizados de Unsloth.


### Paso 4: Preparacion de Dataset para Salida Estructurada (JSON Schema)

Entrenaremos al modelo para que actue como un clasificador corporativo que analiza tickets de soporte y devuelve **JSON estrictamente validable**:


In [10]:
INSTRUCCION = (
    "Analiza el siguiente ticket de soporte y extrae la informacion en formato "
    "JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:"
)

pares = [
    ("Se cayo el servidor principal de produccion en la region US-East! Todos nuestros clientes estan recibiendo error 502!",
     {"categoria": "infraestructura", "urgencia": "critica", "sentimiento": "frustrado", "accion_sugerida": "escalar a SRE guardia y activar cluster de contingencia"}),
    ("Hola, podrian agregar soporte para American Express?",
     {"categoria": "facturacion", "urgencia": "baja", "sentimiento": "positivo", "accion_sugerida": "registrar feature request"}),
    ("Llevo dos semanas esperando respuesta sobre el reporte SOC2. Inaceptable.",
     {"categoria": "cumplimiento", "urgencia": "alta", "sentimiento": "negativo", "accion_sugerida": "asignar Account Manager inmediato"}),
    ("La latencia GraphQL subio un 15%. No es critico pero revisen indices.",
     {"categoria": "rendimiento", "urgencia": "media", "sentimiento": "neutral", "accion_sugerida": "revisar query plan"}),
    ("He detectado multiples intentos de login fallidos desde una IP extrana en Rusia.",
     {"categoria": "seguridad", "urgencia": "critica", "sentimiento": "alerta", "accion_sugerida": "bloquear IP y forzar reset de password"}),
    ("Como puedo cambiar mi direccion de facturacion en el portal?",
     {"categoria": "soporte_usuario", "urgencia": "baja", "sentimiento": "neutral", "accion_sugerida": "enviar guia de configuracion de cuenta"}),
    ("El API de pagos esta devolviendo error 401 a pesar de usar tokens validos.",
     {"categoria": "integracion", "urgencia": "alta", "sentimiento": "preocupado", "accion_sugerida": "revisar servicio de autenticacion de API"}),
    ("Gracias por la ayuda de ayer, el sistema funciona perfecto ahora!",
     {"categoria": "feedback", "urgencia": "informativa", "sentimiento": "muy positivo", "accion_sugerida": "cerrar ticket y agradecer al cliente"}),
]

# ATENCION: estos 42 ejemplos comparten la MISMA salida y son el 84% del
# dataset. Con solo 35 actualizaciones de pesos, el modelo aprendera a
# responder "media/neutral/investigacion estandar" casi siempre.
# Sustituir por tickets reales y variados es la mejora de mayor impacto.
for i in range(42):
    cat = ["seguridad", "infraestructura", "facturacion",
           "rendimiento", "soporte_usuario"][i % 5]
    pares.append((
        f"Ticket sintetico de prueba #{i+5}: Problema reportado en {cat}.",
        {"categoria": cat, "urgencia": "media", "sentimiento": "neutral",
         "accion_sugerida": "investigacion estandar"},
    ))

dataset = Dataset.from_list([{
    "messages": [
        {"role": "user", "content": f"{INSTRUCCION}\n{ticket}"},
        {"role": "assistant", "content": json.dumps(salida, ensure_ascii=False)},
    ]
} for ticket, salida in pares])


def formatear(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}


dataset = dataset.map(formatear, batched=True, remove_columns=["messages"])
print("\n=== EJEMPLO FORMATEADO ===")
print(repr(dataset[0]["text"]))

n_bos = dataset[0]["text"].count(tokenizer.bos_token)
if n_bos != 1:
    print(f"!! {n_bos} tokens BOS (deberia haber 1). Gemma degenera con doble BOS.")

largos = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in dataset["text"]]

print(f"Tokens por ejemplo: min={min(largos)} max={max(largos)} (limite={MAX_SEQ_LENGTH})")
if max(largos) > MAX_SEQ_LENGTH:
    print("!! Hay ejemplos que se truncaran y perderan el cierre del JSON.")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]


=== EJEMPLO FORMATEADO ===
'<bos><start_of_turn>user\nAnaliza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:\nSe cayo el servidor principal de produccion en la region US-East! Todos nuestros clientes estan recibiendo error 502!<end_of_turn>\n<start_of_turn>model\n{"categoria": "infraestructura", "urgencia": "critica", "sentimiento": "frustrado", "accion_sugerida": "escalar a SRE guardia y activar cluster de contingencia"}<end_of_turn>\n'
Tokens por ejemplo: min=85 max=114 (limite=512)


### Paso 5: Entrenamiento Acelerado con `SFTTrainer` y Optimizaciones de Unsloth

Entrenamos con la integracion nativa de Unsloth y `trl.SFTTrainer`:


In [11]:
FastLanguageModel.for_training(model)

training_args = SFTConfig(
    output_dir="./unsloth_gemma_json_output",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,     # batch efectivo = 8
    learning_rate=1e-4,
    num_train_epochs=5,                # 50/8 = 7 pasos/epoca -> 35 en total
    warmup_ratio=0.1,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    bf16=True,                         # Gemma-2 NO tolera fp16
    fp16=False,
    max_grad_norm=0.5,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

collator_sin_mascara = trainer.data_collator
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)
collator_con_mascara = trainer.data_collator


print("\n=== ENTRENAMIENTO ===")
trainer.train()

perdidas = [h["loss"] for h in trainer.state.log_history if "loss" in h]
if perdidas:
    print(f"\nLoss: {perdidas[0]:.3f} -> {perdidas[-1]:.3f}")
    if perdidas[-1] > 2.0:
        print("!! Loss final alto: mas epocas o mejores datos.")

for n, p in model.named_parameters():
    if p.requires_grad and not torch.isfinite(p).all():
        print(f"!! PESO NO FINITO en {n}: adaptadores corruptos.")
        break

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7f245fc1f600>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc1f8d0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc44090>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc44220>
Traceback (most recent call last):
  File "/usr/local/

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7f245fdc1c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc47e70>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc477e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/accelerate/utils/memory.py", line 46, in clear_device_cache
    gc.collect()
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7f245fc47510>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-package


=== ENTRENAMIENTO ===


Step,Training Loss
1,1.916300
2,1.828100
3,1.760300
4,1.705400
5,1.065500
6,0.548900
7,1.137400
8,0.483000
9,0.248500
10,0.308400



Loss: 1.916 -> 0.216


### Paso 6: Inferencia de Alta Velocidad con `FastLanguageModel.for_inference`

Activamos el modo de inferencia acelerado de Unsloth (hasta 2x mas rapido que Hugging Face estandar):


In [12]:
FastLanguageModel.for_inference(model)
FIN_TURNO = tokenizer.convert_tokens_to_ids("<end_of_turn>")


def analizar(ticket):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCCION}\n{ticket}"}],
        tokenize=False, add_generation_prompt=True,
    )
    # add_special_tokens=False es obligatorio: la plantilla ya puso <bos>.
    # Con <bos><bos> Gemma degenera en tokens <unusedN>.
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    salida = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=[tokenizer.eos_token_id, FIN_TURNO],
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        salida[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


pruebas = [
    "Ayuda urgente! Olvide la clave maestra del bucket S3 y mis pipelines se detuvieron!",
    "Buenas tardes, existe documentacion sobre los webhooks de la version 3?",
    "El dashboard tarda 40 segundos en cargar desde el despliegue de ayer.",
    "Detectamos trafico anomalo saliendo de uno de nuestros contenedores.",
]

print("\n=== RESULTADOS ===")
vistos = []
for t in pruebas:
    r = analizar(t)
    print(f"\nTicket: {t}")
    try:
        d = json.loads(r)
        print("JSON:  ", json.dumps(d, ensure_ascii=False))
        vistos.append(d.get("categoria"))
    except json.JSONDecodeError:
        print("NO PARSEA:", repr(r[:200]))

if len(set(vistos)) == 1 and len(vistos) > 1:
    print("\n!! Todas las respuestas dan la misma categoria.")
    print("   No es el entrenamiento: es el dataset. Los 42 ejemplos")
    print("   sinteticos identicos dominan la senal. Escribe 30-40")
    print("   tickets reales y variados.")



=== RESULTADOS ===

Ticket: Ayuda urgente! Olvide la clave maestra del bucket S3 y mis pipelines se detuvieron!
JSON:   {"categoria": "infraestructura", "urgencia": "critica", "sentimiento": "panic", "accion_sugerida": "restablecer bucket s3 y re-configurar pipelines"}

Ticket: Buenas tardes, existe documentacion sobre los webhooks de la version 3?
JSON:   {"categoria": "soporte", "urgencia": "media", "sentimiento": "neutral", "accion_sugerida": "investigacion estandar"}

Ticket: El dashboard tarda 40 segundos en cargar desde el despliegue de ayer.
JSON:   {"categoria": "rendimiento", "urgencia": "media", "sentimiento": "neutral", "accion_sugerida": "investigacion estandar"}

Ticket: Detectamos trafico anomalo saliendo de uno de nuestros contenedores.
JSON:   {"categoria": "seguridad", "urgencia": "critica", "sentimiento": "alarmante", "accion_sugerida": "bloquear contenedor y activar equipo de seguridad"}


### Paso 7: Exportacion Directa a Formato GGUF para Produccion (Ollama)

Unsloth permite guardar el modelo entrenado directamente en formato binario cuantizado **GGUF** con metodo de cuantizacion `q4_k_m` (excelente balance entre tamano y fidelidad):


In [13]:
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.60s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10840-mix-d5c17a0 (app-b10840-mix-d5c17a0-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/gemma-2-2b-it.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model_gguf_gguf/Modelfile


351

In [16]:
!ls -lh model_gguf_gguf/

total 1.6G
-rw-r--r-- 1 root root 1.6G Sep  9 21:55 gemma-2-2b-it.Q4_K_M.gguf
-rw-r--r-- 1 root root  376 Sep  9 21:55 Modelfile


In [18]:
instr = "Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:"

open("Modelfile", "w").write(f'''FROM ./gemma-2-2b-it.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER stop "<end_of_turn>"
TEMPLATE """<start_of_turn>user
{instr}
{{{{ .Prompt }}}}<end_of_turn>
<start_of_turn>model
{{{{ .Response }}}}<end_of_turn>
"""''')

print(open("Modelfile").read())

FROM ./gemma-2-2b-it.Q4_K_M.gguf
PARAMETER temperature 0
PARAMETER stop "<end_of_turn>"
TEMPLATE """<start_of_turn>user
Analiza el siguiente ticket de soporte y extrae la informacion en formato JSON con las claves categoria, urgencia, sentimiento y accion_sugerida:
{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
"""


In [19]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ollama_gemma2_custom
!cp model_gguf_gguf/gemma-2-2b-it.Q4_K_M.gguf /content/drive/MyDrive/ollama_gemma2_custom/
!cp Modelfile /content/drive/MyDrive/ollama_gemma2_custom/
!ls -lh /content/drive/MyDrive/ollama_gemma2_custom/

Mounted at /content/drive
total 1.6G
-rw------- 1 root root 1.6G Sep  9 22:12 gemma-2-2b-it.Q4_K_M.gguf
-rw------- 1 root root  346 Sep  9 22:12 Modelfile


## **En OLLAMA**

- Instala Ollama desde ollama.com/download. Elige tu sistema operativo y sigue el instalador.

- Abre Google Drive en el navegador, entra a ollama_gemma2_custom y descarga los dos archivos: el .gguf de 1.6 GB y el Modelfile.

- Crea una carpeta nueva en tu computadora y mete los dos archivos ahí. Juntos, en la misma carpeta.

- Abre una terminal dentro de esa carpeta y ejecuta:

`ollama create techcloud-classifier -f Modelfile`

- Pruebalo

`ollama run techcloud-classifier "El API de pagos devuelve 401 con tokens validos"`